In [1]:
# Step 1 — audit what's actually downloaded locally
import os

DATA_DIR = r"F:\messi_heatmap\open-data\data"  # adjust if your path differs

# 1. Does Ligue 1 exist locally, and which seasons?
ligue1_matches_dir = os.path.join(DATA_DIR, "matches", "7")
print("Ligue 1 matches folder exists:", os.path.isdir(ligue1_matches_dir))
if os.path.isdir(ligue1_matches_dir):
    print("Season files found:", os.listdir(ligue1_matches_dir))

# 2. Confirm the specific PSG seasons (season_id 108 = 2021/2022, 235 = 2022/2023)
for season_id in [108, 235]:
    path = os.path.join(ligue1_matches_dir, f"{season_id}.json")
    print(f"season_id {season_id} present:", os.path.exists(path))

# 3. Check how many event files exist overall vs how many matches reference them
events_dir = os.path.join(DATA_DIR, "events")
print("\nTotal event files on disk:", len(os.listdir(events_dir)) if os.path.isdir(events_dir) else "MISSING events dir")

# 4. Re-rank your existing season event counts (from your last full extraction)
# adjust path if your CSV landed somewhere else
import pandas as pd
csv_path = "../data/messi_locations_full_career.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\nEvent counts per season (sorted, highest first):")
    print(df.groupby("season_name").size().sort_values(ascending=False))
else:
    print(f"\nCouldn't find {csv_path} — update the path if it's elsewhere")

Ligue 1 matches folder exists: True
Season files found: ['108.json', '235.json', '27.json']
season_id 108 present: True
season_id 235 present: True

Total event files on disk: 4235

Event counts per season (sorted, highest first):
season_name
2014/2015    9986
2011/2012    9930
2010/2011    9542
2020/2021    8941
2019/2020    8130
2017/2018    7830
2018/2019    7457
2012/2013    7398
2009/2010    7392
2015/2016    7370
2016/2017    7191
2008/2009    6660
2013/2014    6244
2007/2008    4789
2006/2007    4655
2005/2006    2097
2022         1548
2018          930
2024          701
2004/2005     179
dtype: int64


In [2]:
# Step 2 — full career extractor, now including PSG
import os, json
import pandas as pd

DATA_DIR = r"F:\messi_heatmap\open-data\data"  # same path as before

TARGET_TEAMS = {"Barcelona", "Argentina", "Paris Saint-Germain"}
PLAYER_MATCH = "Messi"

def load_competitions():
    with open(os.path.join(DATA_DIR, "competitions.json"), encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

def find_relevant_matches():
    comps = load_competitions()
    relevant = []
    for _, row in comps.iterrows():
        comp_id, season_id = row["competition_id"], row["season_id"]
        path = os.path.join(DATA_DIR, "matches", str(comp_id), f"{season_id}.json")
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as f:
            matches = json.load(f)
        for m in matches:
            home = m["home_team"]["home_team_name"]
            away = m["away_team"]["away_team_name"]
            team_hit = home if home in TARGET_TEAMS else (away if away in TARGET_TEAMS else None)
            if team_hit is None:
                continue
            relevant.append({
                "match_id": m["match_id"],
                "match_date": m.get("match_date"),
                "competition_id": comp_id,
                "season_id": season_id,
                "competition_name": row["competition_name"],
                "season_name": row["season_name"],
                "team": team_hit,
            })
    return pd.DataFrame(relevant)

def extract_messi_events(match_row):
    path = os.path.join(DATA_DIR, "events", f"{match_row['match_id']}.json")
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        events = json.load(f)
    rows = []
    for e in events:
        player = e.get("player", {})
        if not player or PLAYER_MATCH not in player.get("name", ""):
            continue
        loc = e.get("location")
        if not loc:
            continue
        rows.append({
            "match_id": match_row["match_id"],
            "match_date": match_row["match_date"],
            "competition_name": match_row["competition_name"],
            "season_name": match_row["season_name"],
            "team": match_row["team"],
            "x": loc[0], "y": loc[1],
            "event_type": e.get("type", {}).get("name"),
            "minute": e.get("minute"),
        })
    return rows

print("Scanning matches (Barcelona / Argentina / PSG)...")
matches_df = find_relevant_matches()
print(f"Found {len(matches_df)} relevant matches.")

all_rows = []
for i, row in matches_df.iterrows():
    all_rows.extend(extract_messi_events(row))
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(matches_df)} processed...")

df = pd.DataFrame(all_rows)
df["x"] = pd.to_numeric(df["x"])
df["y"] = pd.to_numeric(df["y"])
df["match_date"] = pd.to_datetime(df["match_date"])
df = df.sort_values(["match_date", "minute"])

print(f"\nTotal Messi located events: {len(df)}")
print("\nEvent counts per season (sorted):")
print(df.groupby("season_name").size().sort_values(ascending=False))

Scanning matches (Barcelona / Argentina / PSG)...
Found 648 relevant matches.
  200/648 processed...
  400/648 processed...
  600/648 processed...

Total Messi located events: 132159

Event counts per season (sorted):
season_name
2014/2015    9986
2011/2012    9930
2010/2011    9542
2020/2021    8941
2019/2020    8130
2017/2018    7830
2018/2019    7457
2012/2013    7398
2009/2010    7392
2015/2016    7370
2022/2023    7303
2016/2017    7191
2008/2009    6660
2013/2014    6244
2021/2022    5886
2007/2008    4789
2006/2007    4655
2005/2006    2097
2022         1548
2018          930
2024          701
2004/2005     179
dtype: int64


In [12]:
# Step 3 — split highest-volume seasons into first/second half by date
import numpy as np

SPLIT_SEASONS = ["2014/2015", "2011/2012", "2010/2011", "2020/2021"]  # top 4 -> +4 frames = 25 total
# add "2019/2020" to this list if you want 26 instead of 25

def make_season_label(row):
    if row["season_name"] in SPLIT_SEASONS:
        season_dates = df[df["season_name"] == row["season_name"]]["match_date"]
        midpoint = season_dates.min() + (season_dates.max() - season_dates.min()) / 2
        half = "A" if row["match_date"] <= midpoint else "B"
        return f"{row['season_name']} ({half})"
    return row["season_name"]

df["season_label"] = df.apply(make_season_label, axis=1)

print("Frame counts after splitting:")
print(df.groupby("season_label").size().sort_values(ascending=False))
print(f"\nTotal distinct frames: {df['season_label'].nunique()}")

Frame counts after splitting:
season_label
2019/2020        8130
2017/2018        7830
2018/2019        7457
2012/2013        7398
2009/2010        7392
2015/2016        7370
2022/2023        7303
2016/2017        7191
2008/2009        6660
2013/2014        6244
2021/2022        5886
2010/2011 (B)    5531
2014/2015 (B)    5475
2011/2012 (B)    5431
2007/2008        4789
2006/2007        4655
2020/2021 (B)    4605
2014/2015 (A)    4511
2011/2012 (A)    4499
2020/2021 (A)    4336
2010/2011 (A)    4011
2005/2006        2097
2022             1548
2018              930
2023              851
2024              701
2004/2005         179
dtype: int64

Total distinct frames: 27


In [13]:
# Step 4 — chronological ordering (by real date, not label text) + density grids
import numpy as np
from scipy.ndimage import gaussian_filter

# Order every label by its earliest match date — this handles World Cup/Copa America
# slotting in correctly between club seasons automatically
season_order = (
    df.groupby("season_label")["match_date"]
    .min()
    .sort_values()
    .index
    .tolist()
)

print("Chronological order (should be 26):")
for i, s in enumerate(season_order):
    print(f"{i+1:2d}. {s}")

GRID_X, GRID_Y = 140, 100
xi = np.linspace(0, 120, GRID_X)
yi = np.linspace(0, 80, GRID_Y)

def season_density(season_label, sigma=3.0):
    sub = df[df["season_label"] == season_label]
    hist, _, _ = np.histogram2d(sub["x"], sub["y"], bins=[GRID_X, GRID_Y], range=[[0,120],[0,80]])
    hist = hist.T
    smooth = gaussian_filter(hist, sigma=sigma)
    smooth = smooth / smooth.max()
    return smooth

season_grids = {s: season_density(s) for s in season_order}
print("\nBuilt smooth density grids for all", len(season_grids), "frames.")

Chronological order (should be 26):
 1. 2004/2005
 2. 2005/2006
 3. 2006/2007
 4. 2007/2008
 5. 2008/2009
 6. 2009/2010
 7. 2010/2011 (A)
 8. 2010/2011 (B)
 9. 2011/2012 (A)
10. 2011/2012 (B)
11. 2012/2013
12. 2013/2014
13. 2014/2015 (A)
14. 2014/2015 (B)
15. 2015/2016
16. 2016/2017
17. 2017/2018
18. 2018
19. 2018/2019
20. 2019/2020
21. 2020/2021 (A)
22. 2020/2021 (B)
23. 2021/2022
24. 2022/2023
25. 2022
26. 2023
27. 2024

Built smooth density grids for all 27 frames.


In [14]:
# Step 5 — pitch drawing, colormap, and interpolated frame sequence (26 real frames -> smooth morph)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

def draw_pitch(ax):
    """Draw a vertical full pitch (StatsBomb 120x80 coords, plotted as 80 wide x 120 tall)."""
    ax.set_facecolor("black")
    line_color = "#9a9a9a"
    lw = 1.2

    ax.plot([0,80,80,0,0],[0,0,120,120,0], color=line_color, lw=lw)
    ax.plot([0,80],[60,60], color=line_color, lw=lw)
    circle = plt.Circle((40,60), 9.15, color=line_color, fill=False, lw=lw)
    ax.add_patch(circle)
    ax.plot([40],[60], marker='o', color=line_color, markersize=2)

    ax.plot([18,62,62,18,18],[0,0,18,18,0], color=line_color, lw=lw)
    ax.plot([30,50,50,30,30],[0,0,6,6,0], color=line_color, lw=lw)

    ax.plot([18,62,62,18,18],[120,120,102,102,120], color=line_color, lw=lw)
    ax.plot([30,50,50,30,30],[120,120,114,114,120], color=line_color, lw=lw)

    ax.set_xlim(-2, 82)
    ax.set_ylim(-2, 122)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

heat_cmap = "inferno"

# Interpolate between each of the 26 real frames for a smooth morph
season_grids_list = [season_grids[s] for s in season_order]
FRAMES_PER_TRANSITION = 12  # higher = smoother/slower morph

def interpolate_grids(grid_a, grid_b, steps):
    return [grid_a * (1 - t) + grid_b * t for t in np.linspace(0, 1, steps, endpoint=False)]

all_frames = []
frame_labels = []  # human-readable label for the on-screen text (not just a year int anymore)
for i in range(len(season_grids_list) - 1):
    interp = interpolate_grids(season_grids_list[i], season_grids_list[i+1], FRAMES_PER_TRANSITION)
    all_frames.extend(interp)
    # blend the label text proportionally across the transition
    for t in np.linspace(0, 1, FRAMES_PER_TRANSITION, endpoint=False):
        frame_labels.append(season_order[i] if t < 0.5 else season_order[i+1])

all_frames.append(season_grids_list[-1])
frame_labels.append(season_order[-1])

print(f"Total animation frames: {len(all_frames)} (across {len(season_order)} real seasons)")

Total animation frames: 313 (across 27 real seasons)


In [15]:
# # Step 6 — continuous slider position + build/save the final animation
# frame_positions = []
# for i in range(len(season_grids_list) - 1):
#     for t in np.linspace(0, 1, FRAMES_PER_TRANSITION, endpoint=False):
#         frame_positions.append(i + t)
# frame_positions.append(len(season_grids_list) - 1)

# assert len(frame_positions) == len(all_frames) == len(frame_labels)

# fig, ax = plt.subplots(figsize=(6, 9), facecolor="black")
# ax.set_facecolor("black")
# fig.subplots_adjust(top=0.85, bottom=0.18)

# draw_pitch(ax)
# im = ax.imshow(all_frames[0], extent=[0,80,0,120], origin='lower', cmap=heat_cmap, alpha=0.85, zorder=2)
# draw_pitch(ax)  # redraw lines on top of heat

# title_text = fig.text(0.5, 0.93, "Messi's Heatmap", ha='center', color='#d4af6a',
#                        fontsize=22, fontfamily='serif', fontweight='bold')
# label_text = fig.text(0.5, 0.12, season_order[0], ha='center', color='#d4af6a',
#                       fontsize=13, fontfamily='serif')

# POS_MIN, POS_MAX = 0, len(season_order) - 1
# slider_ax = fig.add_axes([0.15, 0.06, 0.7, 0.01])
# slider_ax.set_xlim(POS_MIN, POS_MAX)
# slider_ax.set_ylim(0, 1)
# slider_ax.axis('off')
# slider_line, = slider_ax.plot([POS_MIN, POS_MAX], [0.5, 0.5], color='#d4af6a', lw=1.5)
# slider_dot, = slider_ax.plot([POS_MIN], [0.5], marker='o', color='#d4af6a', markersize=8)
# fig.text(0.15, 0.03, season_order[0], color='#d4af6a', fontsize=8, fontfamily='serif', ha='left')
# fig.text(0.85, 0.03, season_order[-1], color='#d4af6a', fontsize=8, fontfamily='serif', ha='right')

# def update(frame_idx):
#     im.set_data(all_frames[frame_idx])
#     label_text.set_text(frame_labels[frame_idx])
#     slider_dot.set_xdata([frame_positions[frame_idx]])
#     return im, label_text, slider_dot

# ani = animation.FuncAnimation(fig, update, frames=len(all_frames), interval=60, blit=False)

# import os
# os.makedirs("../output", exist_ok=True)
# ani.save("../output/messi_heatmap_full_career.gif", writer="pillow", fps=20, dpi=120,
#           savefig_kwargs={"facecolor": "black"})
# print("Saved ../output/messi_heatmap_full_career.gif")

In [18]:
# Rebuild frame_positions to match the current (updated) season_grids_list length
frame_positions = []
for i in range(len(season_grids_list) - 1):
    for t in np.linspace(0, 1, FRAMES_PER_TRANSITION, endpoint=False):
        frame_positions.append(i + t)
frame_positions.append(len(season_grids_list) - 1)

print(f"frame_positions length: {len(frame_positions)}")
print(f"all_frames length: {len(all_frames)}")
print(f"frame_labels length: {len(frame_labels)}")
# these three should all match exactly
assert len(frame_positions) == len(all_frames) == len(frame_labels)
print("All three arrays match — safe to re-run Step 7.")

frame_positions length: 313
all_frames length: 313
frame_labels length: 313
All three arrays match — safe to re-run Step 7.


In [19]:
# Step 7 (revised) — 5 exports across 3 folders: gif/, video/, excel/
import os
import matplotlib.pyplot as plt
import matplotlib.animation as animation

os.makedirs("../output/gif", exist_ok=True)
os.makedirs("../output/video", exist_ok=True)
os.makedirs("../output/excel", exist_ok=True)

HIGH_DPI = 200  # shared resolution for all 4 visual exports

# =========================================================
# 1 & 2 — VERTICAL (gif + mp4), reusing draw_pitch / all_frames from Step 5-6
# =========================================================
fig_v, ax_v = plt.subplots(figsize=(6, 9), facecolor="black")
ax_v.set_facecolor("black")
fig_v.subplots_adjust(top=0.85, bottom=0.18)

draw_pitch(ax_v)
im_v = ax_v.imshow(all_frames[0], extent=[0,80,0,120], origin='lower', cmap=heat_cmap, alpha=0.85, zorder=2)
draw_pitch(ax_v)

title_v = fig_v.text(0.5, 0.93, "Messi's Heatmap", ha='center', color='#d4af6a',
                      fontsize=22, fontfamily='serif', fontweight='bold')
label_v = fig_v.text(0.5, 0.12, season_order[0], ha='center', color='#d4af6a',
                      fontsize=13, fontfamily='serif')

POS_MIN, POS_MAX = 0, len(season_order) - 1
slider_ax_v = fig_v.add_axes([0.15, 0.06, 0.7, 0.01])
slider_ax_v.set_xlim(POS_MIN, POS_MAX); slider_ax_v.set_ylim(0, 1); slider_ax_v.axis('off')
slider_ax_v.plot([POS_MIN, POS_MAX], [0.5, 0.5], color='#d4af6a', lw=1.5)
slider_dot_v, = slider_ax_v.plot([POS_MIN], [0.5], marker='o', color='#d4af6a', markersize=8)
fig_v.text(0.15, 0.03, season_order[0], color='#d4af6a', fontsize=8, fontfamily='serif', ha='left')
fig_v.text(0.85, 0.03, season_order[-1], color='#d4af6a', fontsize=8, fontfamily='serif', ha='right')

def update_v(frame_idx):
    im_v.set_data(all_frames[frame_idx])
    label_v.set_text(frame_labels[frame_idx])
    slider_dot_v.set_xdata([frame_positions[frame_idx]])
    return im_v, label_v, slider_dot_v

ani_v = animation.FuncAnimation(fig_v, update_v, frames=len(all_frames), interval=60, blit=False)

ani_v.save("../output/gif/messi_heatmap_vertical.gif", writer="pillow", fps=20, dpi=HIGH_DPI,
           savefig_kwargs={"facecolor": "black"})
print("Saved ../output/gif/messi_heatmap_vertical.gif")

try:
    ani_v.save("../output/video/messi_heatmap_vertical.mp4", writer="ffmpeg", fps=20, dpi=HIGH_DPI,
               savefig_kwargs={"facecolor": "black"})
    print("Saved ../output/video/messi_heatmap_vertical.mp4")
except Exception as e:
    print(f"Vertical MP4 failed (need ffmpeg: conda install -c conda-forge ffmpeg): {e}")

plt.close(fig_v)

# =========================================================
# 3 & 4 — HORIZONTAL (gif + mp4)
# =========================================================
def draw_pitch_horizontal(ax):
    ax.set_facecolor("black")
    line_color = "#9a9a9a"; lw = 1.2
    ax.plot([0,120,120,0,0],[0,0,80,80,0], color=line_color, lw=lw)
    ax.plot([60,60],[0,80], color=line_color, lw=lw)
    circle = plt.Circle((60,40), 9.15, color=line_color, fill=False, lw=lw)
    ax.add_patch(circle)
    ax.plot([60],[40], marker='o', color=line_color, markersize=2)
    ax.plot([0,18,18,0],[18,18,62,62], color=line_color, lw=lw)
    ax.plot([0,6,6,0],[30,30,50,50], color=line_color, lw=lw)
    ax.plot([120,102,102,120],[18,18,62,62], color=line_color, lw=lw)
    ax.plot([120,114,114,120],[30,30,50,50], color=line_color, lw=lw)
    ax.set_xlim(-2, 122); ax.set_ylim(-2, 82)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

all_frames_horizontal = [g.T for g in all_frames]

fig_h, ax_h = plt.subplots(figsize=(9, 6), facecolor="black")
ax_h.set_facecolor("black")
fig_h.subplots_adjust(top=0.82, bottom=0.22)

draw_pitch_horizontal(ax_h)
im_h = ax_h.imshow(all_frames_horizontal[0], extent=[0,120,0,80], origin='lower', cmap=heat_cmap, alpha=0.85, zorder=2)
draw_pitch_horizontal(ax_h)

title_h = fig_h.text(0.5, 0.90, "Messi's Heatmap", ha='center', color='#d4af6a',
                      fontsize=20, fontfamily='serif', fontweight='bold')
label_h = fig_h.text(0.5, 0.14, season_order[0], ha='center', color='#d4af6a',
                      fontsize=13, fontfamily='serif')

slider_ax_h = fig_h.add_axes([0.15, 0.08, 0.7, 0.01])
slider_ax_h.set_xlim(POS_MIN, POS_MAX); slider_ax_h.set_ylim(0, 1); slider_ax_h.axis('off')
slider_ax_h.plot([POS_MIN, POS_MAX], [0.5, 0.5], color='#d4af6a', lw=1.5)
slider_dot_h, = slider_ax_h.plot([POS_MIN], [0.5], marker='o', color='#d4af6a', markersize=8)
fig_h.text(0.15, 0.05, season_order[0], color='#d4af6a', fontsize=8, fontfamily='serif', ha='left')
fig_h.text(0.85, 0.05, season_order[-1], color='#d4af6a', fontsize=8, fontfamily='serif', ha='right')

def update_h(frame_idx):
    im_h.set_data(all_frames_horizontal[frame_idx])
    label_h.set_text(frame_labels[frame_idx])
    slider_dot_h.set_xdata([frame_positions[frame_idx]])
    return im_h, label_h, slider_dot_h

ani_h = animation.FuncAnimation(fig_h, update_h, frames=len(all_frames_horizontal), interval=60, blit=False)

ani_h.save("../output/gif/messi_heatmap_horizontal.gif", writer="pillow", fps=20, dpi=HIGH_DPI,
           savefig_kwargs={"facecolor": "black"})
print("Saved ../output/gif/messi_heatmap_horizontal.gif")

try:
    ani_h.save("../output/video/messi_heatmap_horizontal.mp4", writer="ffmpeg", fps=20, dpi=HIGH_DPI,
               savefig_kwargs={"facecolor": "black"})
    print("Saved ../output/video/messi_heatmap_horizontal.mp4")
except Exception as e:
    print(f"Horizontal MP4 failed (need ffmpeg: conda install -c conda-forge ffmpeg): {e}")

plt.close(fig_h)

# =========================================================
# 5 — EXCEL export of the input data (raw filtered events + season summary)
# =========================================================
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

excel_path = "../output/excel/messi_input_data.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Raw_Events", index=False)
    summary = df.groupby("season_label").size().reindex(season_order).rename("event_count").reset_index()
    summary.to_excel(writer, sheet_name="Season_Summary", index=False)

# light formatting pass: bold headers, borders, alternating row shading
from openpyxl import load_workbook
wb = load_workbook(excel_path)
thin = Side(style="thin", color="999999")
border = Border(left=thin, right=thin, top=thin, bottom=thin)
fill_alt = PatternFill(start_color="EFEFEF", end_color="EFEFEF", fill_type="solid")

for sheet in wb.sheetnames:
    ws = wb[sheet]
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.border = border
        cell.alignment = Alignment(horizontal="center")
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        for cell in row:
            cell.border = border
            if row_idx % 2 == 0:
                cell.fill = fill_alt
    for col in ws.columns:
        max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 40)

wb.save(excel_path)
print(f"Saved {excel_path} ({len(df)} raw events, {len(summary)} season rows)")

Saved ../output/gif/messi_heatmap_vertical.gif
Saved ../output/video/messi_heatmap_vertical.mp4
Saved ../output/gif/messi_heatmap_horizontal.gif
Saved ../output/video/messi_heatmap_horizontal.mp4
Saved ../output/excel/messi_input_data.xlsx (133010 raw events, 27 season rows)


In [11]:
# Step 9a — find MLS locally and check what's there
import os, json
import pandas as pd

DATA_DIR = r"F:\messi_heatmap\open-data\data"

comps = pd.read_json(os.path.join(DATA_DIR, "competitions.json"))
mls_rows = comps[comps["competition_name"].str.contains("Major League Soccer", case=False, na=False)]
print("MLS entries found locally:")
print(mls_rows[["competition_id", "season_id", "season_name"]])

# Check whether the actual match/event files exist for each MLS season found
for _, row in mls_rows.iterrows():
    comp_id, season_id = row["competition_id"], row["season_id"]
    matches_path = os.path.join(DATA_DIR, "matches", str(comp_id), f"{season_id}.json")
    exists = os.path.exists(matches_path)
    print(f"\ncompetition_id={comp_id}, season_id={season_id} ({row['season_name']}) -> matches file exists: {exists}")
    if exists:
        with open(matches_path, encoding="utf-8") as f:
            matches = json.load(f)
        miami_matches = [m for m in matches if "Miami" in m["home_team"]["home_team_name"] or "Miami" in m["away_team"]["away_team_name"]]
        print(f"  Inter Miami matches in this season: {len(miami_matches)}")

import os
match_ids = [3877115, 3877170, 3877060, 3877090, 3877194, 3877072]
for mid in match_ids:
    path = os.path.join(DATA_DIR, "events", f"{mid}.json")
    print(mid, "exists:", os.path.exists(path))

# Step 9b — re-run extraction with Inter Miami included
TARGET_TEAMS = {"Barcelona", "Argentina", "Paris Saint-Germain", "Inter Miami"}
PLAYER_MATCH = "Messi"

matches_df = find_relevant_matches()  # reuse the function from Step 2
print(f"Found {len(matches_df)} relevant matches (now including MLS).")

all_rows = []
for i, row in matches_df.iterrows():
    all_rows.extend(extract_messi_events(row))

df = pd.DataFrame(all_rows)
df["x"] = pd.to_numeric(df["x"])
df["y"] = pd.to_numeric(df["y"])
df["match_date"] = pd.to_datetime(df["match_date"])
df = df.sort_values(["match_date", "minute"])

print(f"\nTotal Messi located events: {len(df)}")
print("\nEvent counts per season (sorted):")
print(df.groupby("season_name").size().sort_values(ascending=False))

MLS entries found locally:
    competition_id  season_id season_name
64              44        107        2023

competition_id=44, season_id=107 (2023) -> matches file exists: True
  Inter Miami matches in this season: 6
3877115 exists: True
3877170 exists: True
3877060 exists: True
3877090 exists: True
3877194 exists: True
3877072 exists: True
Found 654 relevant matches (now including MLS).

Total Messi located events: 133010

Event counts per season (sorted):
season_name
2014/2015    9986
2011/2012    9930
2010/2011    9542
2020/2021    8941
2019/2020    8130
2017/2018    7830
2018/2019    7457
2012/2013    7398
2009/2010    7392
2015/2016    7370
2022/2023    7303
2016/2017    7191
2008/2009    6660
2013/2014    6244
2021/2022    5886
2007/2008    4789
2006/2007    4655
2005/2006    2097
2022         1548
2018          930
2023          851
2024          701
2004/2005     179
dtype: int64
